# 8교시. 실무 적용 시나리오 설계 및 최종 정리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/08_business_application.ipynb)

**이번 교시 행동:** 견적서·신청서·거래명세서 실물 사진을 비교하고 첫 PoC 한 가지를 고릅니다.

**통과 증거:** `course_outputs/poc_candidate_card.md`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
import base64
import io
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = lambda image: None

EXTENSION_IMAGES = {'quotation': '...', 'application': '...', 'transaction_statement': '...'}
EXTENSION_EXAMPLES = {'quotation': {'name': '견적서', 'fields': ['문서번호', '공급자', '수신', '견적일', '품목', '총액'], 'rules': ['수량×단가=품목금액', '공급가액+부가세=총액'], 'risk': '총액 오류는 구매 의사결정에 직접 영향'}, 'application': {'name': '신청서', 'fields': ['신청번호', '신청자', '소속', '신청 과정', '승인'], 'rules': ['필수 동의', '관리자 승인 상태'], 'risk': '개인정보와 승인 누락을 사람이 확인'}, 'transaction_statement': {'name': '거래명세서', 'fields': ['문서번호', '공급자', '거래일', '품목', '세액', '총액'], 'rules': ['품목 합계=공급가액', '공급가액+세액=총액'], 'risk': '표 행·열 대응이 어긋나면 정산 오류'}}
for key, payload in EXTENSION_IMAGES.items():
    image = Image.open(io.BytesIO(base64.b64decode(payload))).convert("RGB")
    image.thumbnail((320, 400))
    print(key, image.size)
    display(image)


## 형식이 바뀌면 생기는 어려움

- **Excel**: 수식, 병합 셀, 숨김 시트, 숫자 서식
- **Word**: 머리글, 텍스트박스, 변경 추적, 이미지로 삽입된 본문
- **PDF**: 텍스트·스캔 혼합 페이지, 암호, 깨진 문자맵
- **PPT**: 그룹 도형, 읽기 순서, 발표자 노트
- **표 캡처**: 셀 관계가 사라져 행·열 위상을 다시 복원해야 함


In [ ]:
from textwrap import dedent

candidate = "transaction_statement"
example = EXTENSION_EXAMPLES[candidate]
score = {
    "반복량": 4,
    "필드 안정성": 4,
    "오류 영향": 2,
    "예외 빈도": 3,
    "사람 검토 가능성": 5,
}
recommendation = (
    "GO_SMALL"
    if score["반복량"] >= 4 and score["사람 검토 가능성"] >= 4
    else "REVIEW"
)
card = f'''# 문서 자동화 PoC 후보 카드

| 항목 | 내용 |
| --- | --- |
| 선택 문서 | {example["name"]} |
| 추출 필드 | {", ".join(example["fields"])} |
| 검증 규칙 | {" / ".join(example["rules"])} |
| 틀렸을 때 영향 | {example["risk"]} |
| 입력 제한 | 승인된 비식별 한 장 |
| 최종 산출물 | 사람 승인 후 Excel |
| 제안 | {recommendation} |

## 첫 PoC 통과 기준

- 같은 양식 30장을 모아 정답표와 비교한다.
- 필드별 정확도뿐 아니라 수정률과 처리시간을 기록한다.
- 오류 시 자동 저장하지 않고 검토 대기열로 보낸다.
- 개인정보·보존·삭제 정책을 먼저 승인받는다.
'''
output_path = OUTPUT_DIR / "poc_candidate_card.md"
output_path.write_text(dedent(card), encoding="utf-8")
print(dedent(card))
print("CHECKPOINT 1/1 PASS:", output_path)
